# Sommelier — Vietnamese UV Isolated Venv Test on Kaggle (2x T4 GPU)

Runs the sommelier podcast pipeline inside a **clean isolated `venv`** (`/kaggle/working/sommelier_env`) using **`uv`** on Kaggle with dual T4 GPUs.
This notebook dynamically patches runtime dependencies and handles Kaggle paths without modifying repository source files.


## 1. Sanity check the Kaggle runtime & GPUs

In [ ]:
!nvidia-smi
!python --version
!df -h /kaggle/working 2>/dev/null || df -h .
import torch
print('PyTorch version:', torch.__version__)
print('CUDA version:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('Device count (GPUs):', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}:', torch.cuda.get_device_name(i))


## 2. Setup Working Directory & Repository

In [ ]:
import os, sys, glob, pathlib

# Kaggle dynamic working directory setup
BASE_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else os.getcwd()
PROJECT_DIR = os.path.join(BASE_DIR, 'sommelier')
ENV_DIR = os.path.join(BASE_DIR, 'sommelier_env')
AUDIO_DIR = os.path.join(BASE_DIR, 'vi_audio')

print(f"BASE_DIR: {BASE_DIR}")
print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"ENV_DIR: {ENV_DIR}")
print(f"AUDIO_DIR: {AUDIO_DIR}")

# Clone repository if running in a clean Kaggle environment
if not os.path.exists(PROJECT_DIR):
    os.chdir(BASE_DIR)
    !git clone https://github.com/tuanad121/sommelier.git
else:
    print(f"Repository already exists at {PROJECT_DIR}")


## 3. Install dependencies into isolated venv via UV (~10 min)

Mirrors the three-step install order (torch CUDA 12.6 first, then requirements).


In [ ]:
# 1. Install uv package manager
!pip install -q uv yt-dlp

# 2. Create clean isolated virtual environment
!uv venv --allow-existing {ENV_DIR}

# 3. Define proposed optimized dependencies list
PROPOSED_REQUIREMENTS = """
numpy==1.26.4
torch==2.7.1
torchaudio==2.7.1
torchvision==0.22.1
lightning==2.3.3
torchmetrics==1.7.4
onnxruntime-gpu>=1.19.0
nemo-toolkit[asr]==3.0.0
pyannote.audio==4.0.7
speechbrain==1.0.3
faster-whisper==1.2.1
whisperx==3.3.1
ctranslate2>=4.4.0
demucs>=4.0.0
panns-inference
librosa==0.11.0
soundfile==0.13.1
pydub==0.25.1
julius==0.2.7
numba==0.61.2
transformers==4.53.0
huggingface-hub==0.33.4
g2pk
jamo
nltk==3.9.1
openai==1.97.1
tritony==0.0.20
tritonclient[all]
pandas==2.3.1
PyYAML==6.0.2
tqdm==4.67.1
wandb==0.21.0
requests==2.32.4
einops==0.8.1
hydra-core==1.3.2
omegaconf==2.3.0
yt-dlp
setuptools>=79.0.0
sacrebleu
"""

req_file = os.path.join(BASE_DIR, 'requirements_proposed.txt')
with open(req_file, 'w') as f:
    f.write(PROPOSED_REQUIREMENTS.strip())

print('>>> Step 1: Pre-installing PyTorch CUDA 12.6 inside isolated venv...')
!uv pip install --python {ENV_DIR} torch==2.7.1 torchaudio==2.7.1 torchvision==0.22.1 --extra-index-url https://download.pytorch.org/whl/cu126

print('\n>>> Step 2: Installing proposed dependencies into venv via UV...')
!uv pip install --python {ENV_DIR} -r {req_file} --extra-index-url https://download.pytorch.org/whl/cu126 --index-strategy unsafe-best-match


In [ ]:
!uv pip uninstall --python {ENV_DIR} nvidia-resiliency-ext


In [ ]:
# Verify imports inside isolated venv
python_bin = os.path.join(ENV_DIR, 'bin', 'python')
!{python_bin} -c "import torch, whisperx, demucs, nemo, pyannote.audio, transformers; from chunkformer import ChunkFormerModel; print('vEnv PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available(), 'GPUs:', torch.cuda.device_count()); print('transformers:', transformers.__version__); print('nemo:', nemo.__version__)"


## 4. Hugging Face authentication & Config update

In [ ]:
# Authenticate HuggingFace Token (supports Kaggle Secrets & Interactive Input)
import json, pathlib

hf_token = ""
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    print("Fetched HF_TOKEN from Kaggle Secrets.")
except Exception as e:
    print("Kaggle Secrets not available or HF_TOKEN secret missing.")

if not hf_token:
    from getpass import getpass
    hf_token = getpass('Enter HF Token (hf_...): ')

from huggingface_hub import login
login(token=hf_token)

# Update config.json in podcast-pipeline
cfg_path = os.path.join(PROJECT_DIR, 'podcast-pipeline', 'config.json')
if os.path.exists(cfg_path):
    cfg = json.loads(pathlib.Path(cfg_path).read_text())
    cfg['huggingface_token'] = hf_token
    pathlib.Path(cfg_path).write_text(json.dumps(cfg, indent=2, ensure_ascii=False))
    print(f'Updated Hugging Face token in {cfg_path}')


## 5. Prepare Audio Input (Auto-detects Kaggle Dataset `/kaggle/input`, Drag-Drop, or Fallback)

In [ ]:
import os, glob, random, shutil, pathlib
import torch, torchaudio

# Always prepare a writable working audio folder
BASE_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else os.getcwd()
AUDIO_DIR = os.path.join(BASE_DIR, 'vi_audio')
os.makedirs(AUDIO_DIR, exist_ok=True)

# 1. Auto-detect if user attached audio via Kaggle Datasets (/kaggle/input/...)
input_audio_files = []
if os.path.exists('/kaggle/input'):
    extensions = ('*.mp3', '*.wav', '*.flac', '*.m4a', '*.aac', '*.ogg')
    for ext in extensions:
        input_audio_files.extend(glob.glob(f'/kaggle/input/**/{ext}', recursive=True))

if input_audio_files:
    print(f"✅ Detected {len(input_audio_files)} audio file(s) in /kaggle/input/:")
    for f in input_audio_files:
        print("  -", f)
        dst = os.path.join(AUDIO_DIR, os.path.basename(f))
        if not os.path.exists(dst):
            shutil.copy(f, dst)
    print(f"Sync completed into writable folder: {AUDIO_DIR}")

# 2. Scan audio files in /kaggle/working/vi_audio/
audio_extensions = ('.mp3', '.wav', '.flac', '.m4a', '.aac', '.ogg')
audio_files = [f for f in glob.glob(os.path.join(AUDIO_DIR, '*')) if f.lower().endswith(audio_extensions)]

# 3. Fallback: If no audio files exist, generate a 10s synthetic test WAV file
if not audio_files:
    print('\n⚠️ No audio files found in /kaggle/input or /kaggle/working/vi_audio/.')
    print('Generating synthetic benchmark WAV file in vi_audio for smoke testing...')
    sample_rate = 16000
    duration_sec = 10
    t = torch.linspace(0, duration_sec, sample_rate * duration_sec)
    waveform = 0.3 * torch.sin(2 * 3.14159 * 440 * t) + 0.2 * torch.sin(2 * 3.14159 * 880 * t)
    waveform = waveform.unsqueeze(0)
    
    fallback_wav = os.path.join(AUDIO_DIR, 'kaggle_test_sample.wav')
    torchaudio.save(fallback_wav, waveform, sample_rate)
    audio_files = [fallback_wav]
    print(f'✅ Fallback test WAV created at: {fallback_wav}')

print(f'\n✅ Total audio file(s) ready for pipeline: {len(audio_files)}')
for af in audio_files:
    print('  -', af)


## 6. Dynamic In-Notebook Patches & Run Pipeline on 2x T4 GPUs

In [ ]:
# Dynamically apply in-notebook patches without altering original repo files outside this execution
import glob, os

# Patch 1: Dynamically patch pkg_resources in venv site-packages regardless of Python version
site_pkgs = glob.glob(os.path.join(ENV_DIR, 'lib', 'python*', 'site-packages'))
if site_pkgs:
    pkg_res_file = os.path.join(site_pkgs[0], 'pkg_resources.py')
    with open(pkg_res_file, 'w', encoding='utf-8') as f:
        f.write('def declare_namespace(name): pass\n')
    print(f'✅ Dynamic patch applied to {pkg_res_file}')

# Patch 2: Dynamically patch main_original_ASR_MoE.py for PyTorch 2.x weights_only compatibility
main_script = os.path.join(PROJECT_DIR, 'podcast-pipeline', 'main_original_ASR_MoE.py')
if os.path.exists(main_script):
    with open(main_script, 'r', encoding='utf-8') as f:
        code = f.read()
    
    target_old = 'def _patched_load(path_or_url: Union[IO, str, Path], map_location=None) -> Any:'
    target_new = 'def _patched_load(path_or_url: Union[IO, str, Path], map_location=None, weights_only=False, **kwargs) -> Any:'
    
    if target_old in code:
        code = code.replace(target_old, target_new)
        with open(main_script, 'w', encoding='utf-8') as f:
            f.write(code)
        print(f'✅ Dynamic patch applied to {main_script}')
    else:
        print(f'ℹ️ {main_script} already patched or line signature differs.')
    
    target_import = "from chunkformer import ChunkFormerModel"
    if target_import in code:
        code = code.replace(target_import, f"# {target_import}")
        with open(main_script, "w", encoding="utf-8") as f:
            f.write(code)
        print(f"✅ Dynamic patch applied to remove chunkformer import in {main_script}")
    
# Patch 6: Fix WhisperX 3.8.6 missing whisperx.types module
whisper_asr_script = os.path.join(PROJECT_DIR, "podcast-pipeline", "models", "whisper_asr.py")
if os.path.exists(whisper_asr_script):
    with open(whisper_asr_script, "r", encoding="utf-8") as f:
        wasr_code = f.read()
    old_imp = "from whisperx.types import TranscriptionResult, SingleSegment"
    if old_imp in wasr_code and "try:" not in wasr_code.split(old_imp)[0][-10:]:
        new_imp = "try:\\n    from whisperx.types import TranscriptionResult, SingleSegment\\nexcept (ImportError, ModuleNotFoundError):\\n    TranscriptionResult = dict\\n    SingleSegment = dict"
        wasr_code = wasr_code.replace(old_imp, new_imp)
        with open(whisper_asr_script, "w", encoding="utf-8") as f:
            f.write(wasr_code)
        print(f"✅ Patch 6: fixed whisperx.types import in {whisper_asr_script}")
    
# Patch 7: Fix pyannote 4.x API change use_auth_token -> token
if os.path.exists(main_script):
    with open(main_script, "r", encoding="utf-8") as f:
        code = f.read()
    if "use_auth_token=" in code:
        code = code.replace("use_auth_token=", "token=")
        with open(main_script, "w", encoding="utf-8") as f:
            f.write(code)
        print(f"✅ Patch 7: use_auth_token -> token in {main_script}")


In [ ]:
# Run the pipeline configured for Kaggle 2x T4 GPU
os.chdir(os.path.join(PROJECT_DIR, 'podcast-pipeline'))
python_bin = os.path.join(ENV_DIR, 'bin', 'python')
import sys
site_packages = os.path.join(ENV_DIR, 'lib', f'python{sys.version_info.major}.{sys.version_info.minor}', 'site-packages')
nvidia_lib = f"{site_packages}/nvidia/cudnn/lib:{site_packages}/torch/lib"

import os
os.environ["LD_LIBRARY_PATH"] = os.environ.get("LD_LIBRARY_PATH", "") + ":" + nvidia_lib

!CUDA_VISIBLE_DEVICES=0,1 {python_bin} main_original_ASR_MoE.py \
  --input_folder_path {AUDIO_DIR} \
  --lang vi \
  --vad \
  --dia3 \
  --ASRMoE \
  --no-demucs \
  --whisperx_word_timestamps \
  --no-qwen3omni \
  --no-sepreformer \
  --LLM case_0 \
  --seg_th 0.11 \
  --min_cluster_size 11 \
  --clust_th 0.5 \
  --merge_gap 2


## 7. Inspect Output

In [ ]:
import glob, json, pathlib

result_folder = os.path.join(AUDIO_DIR, '_final')
json_paths = sorted(glob.glob(f'{result_folder}/**/*.json', recursive=True))
print(f'Found {len(json_paths)} result file(s):')
for p in json_paths:
    print(' -', p)

if json_paths:
    result = json.loads(pathlib.Path(json_paths[0]).read_text())
    print('\nMetadata:')
    print(json.dumps(result.get('metadata', {}), indent=2, ensure_ascii=False))
    print(f"\nFirst 3 segments of {len(result.get('segments', []))}:")
    for seg in result.get('segments', [])[:3]:
        print(json.dumps(seg, indent=2, ensure_ascii=False))


## 8. Display Results in a DataFrame (Speaker, Time, Audio, Text)

In [ ]:
import pandas as pd

if json_paths:
    result = json.loads(pathlib.Path(json_paths[0]).read_text())
    audio_name = result.get("metadata", {}).get("audio_name", pathlib.Path(json_paths[0]).stem)
    
    rows = []
    for seg in result.get("segments", []):
        # Format time
        start = seg.get("start", 0)
        end = seg.get("end", 0)
        time_str = f"{start:.2f} - {end:.2f}"
        
        # Get speaker
        speaker = seg.get("speaker", "Unknown")
        
        # Get text (fallback to whisper or ensemble if text is not available)
        text = seg.get("text") or seg.get("text_ensemble") or seg.get("text_whisper") or ""
        
        rows.append({
            "Speaker": speaker,
            "Time": time_str,
            "Audio": audio_name,
            "Text": text
        })
    
    df = pd.DataFrame(rows)
    # Configure pandas display to show full text
    pd.set_option("display.max_colwidth", None)
    display(df)
else:
    print("No results found to display.")


## Kaggle 2x T4 Troubleshooting & Tips

- **Dual GPU Allocation**: Specified `CUDA_VISIBLE_DEVICES=0,1` for multi-GPU runtime.
- **Kaggle Secrets**: Set `HF_TOKEN` in Kaggle Secrets (Add-ons -> Secrets) so Hugging Face models auto-authenticate.
- **Input Audio**: Supports Kaggle Datasets (`/kaggle/input/...`), direct drag-drop (`/kaggle/working/vi_audio/`), or synthetic benchmark fallback audio.
- **Isolated Venv**: `uv` isolates dependencies in `/kaggle/working/sommelier_env` to avoid Kaggle pre-installed package conflicts.
